# Lesson 20: Designing AI systems

*Worked solutions for [the lesson page](../../lessons/20-system-design.html). The exercise version without solutions is `../20-system-design.ipynb`.*

Pull the whole course together into a repeatable way of designing AI systems, apply it to three worked case studies, and learn to read papers and stay current without drowning.

**Runs on:** Any computer. This notebook is a design-document workbook.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy

## A design framework

The framework has eight parts; the lesson walks through them. Only part 7, operations, has code: a back-of-envelope estimator. Keep it and reuse it for every design you write.

### 7. Operations

Illustrative prices; plug in your provider's current ones and your own measurements of time to first token and tokens per second.

In [ ]:
def estimate(requests_per_day, in_tokens, out_tokens, price_in, price_out,
             cached_fraction=0.0, cache_discount=0.9,
             batch_fraction=0.0, batch_discount=0.5,
             ttft_s=0.6, tokens_per_s=60):
    """Monthly cost and typical latency for one model call per request.

    Prices are per million tokens. cached_fraction is the share of input
    tokens read from the prompt cache at (1 - cache_discount) of the price.
    batch_fraction is the share of requests sent through a batch API at
    (1 - batch_discount) of the price."""
    in_price = price_in * ((1 - cached_fraction)
                           + cached_fraction * (1 - cache_discount))
    per_request = in_tokens / 1e6 * in_price + out_tokens / 1e6 * price_out
    per_request *= (1 - batch_fraction) + batch_fraction * (1 - batch_discount)
    latency = ttft_s + out_tokens / tokens_per_s
    return {"per_request": round(per_request, 4),
            "monthly": round(per_request * requests_per_day * 30),
            "latency_s": round(latency, 1)}

# Illustrative prices; plug in your provider's current ones.
base = dict(requests_per_day=2000, in_tokens=4000, out_tokens=350,
            price_in=3.0, price_out=15.0)
print("plain       ", estimate(**base))
print("cached 70%  ", estimate(**base, cached_fraction=0.7))
print("+ batch 50% ", estimate(**base, cached_fraction=0.7, batch_fraction=0.5))

### Hands-on: choose the approach for six briefs

For each brief, write three lines: the rung or tool you would start with, why the one below it is not enough, and the first metric you would measure. Decide in under four minutes per brief; that is roughly the time you get in an interview.

1. A warehouse wants to know how many units of each of 2,000 products it will ship next week.
2. A laboratory wants to flag microscope images that contain a contaminated sample, 50,000 images a day.
3. A school must build a timetable that satisfies about 300 constraints on rooms, teachers and periods.
4. An insurer wants to decide which of 10,000 daily claims to send for manual review, and has five years of labelled history.
5. A support team wants an assistant that answers questions about a 500-page policy manual that changes monthly, and hands over to a person when unsure.
6. A platform team wants an assistant that fixes failing builds by reading the logs, editing the code and rerunning the tests.

You are done when every brief has a tool, a reason and a metric, and at most two of the six use a language model as the core.

#### Solution

1. **Forecasting** (time series, recommenders and graphs lesson): a seasonal naive baseline per product, then gradient boosting on lag and calendar features if it beats the baseline. A language model has no advantage on 2,000 numeric series. Metric: error against the seasonal baseline, measured on the last eight weeks held out.
2. **Vision classifier** (computer vision lesson): a small CNN or a fine-tuned vision transformer on labelled images. A vision-language model could label the first few thousand examples, but at 50,000 images a day it is too slow and costly to be the classifier. Metric: recall at a threshold that keeps false alarms reviewable.
3. **Constraint solver** (search, planning and classical AI lesson): the problem is exactly a constraint satisfaction problem. No learning is needed; a solver gives a valid timetable or proves there is none. Metric: constraints satisfied, and solve time.
4. **Classical ML** (classical machine learning lesson): gradient boosting on the claim features, with labelled history as training data. An LLM may help extract features from free-text claim notes, but the decision model is tabular. Metric: recall on fraudulent or erroneous claims at the review capacity you have.
5. **LLM with retrieval** (RAG lesson): the knowledge changes monthly and answers must cite the manual, so retrieval beats fine-tuning and long context. Prompting alone fails because the model has not read the manual. Metric: citation accuracy and abstention rate on a question set.
6. **Agent with a reasoning model** (agents and reasoning models lessons): the steps depend on what the logs say, so a fixed workflow will not do, and diagnosis across several files benefits from a thinking budget. Guard it with step and cost limits, a sandbox and tests that must pass. Metric: fraction of builds fixed with tests green, and cost per fix.

Two of six have a language model at the core. That ratio is typical of real organisations, and it is why part IX is in this course.

## Case studies

Read the three case studies and the design drill in the lesson. They share one pattern: classical methods or specialised models for numbers, detection, images and ranking; language models for language, synthesis and interaction.

### Hands-on: design reviews

Write a two-page design document, using the eight-part framework, for **one** of these:

1. A meeting assistant that records project meetings, produces minutes with action items, and answers questions across all past meetings.
2. A code-review assistant for your team's repository that comments on pull requests.
3. An assistant that helps scientists find, compare and summarise papers relevant to their research.
4. A recommendation feed for a hobbyist marketplace with 50,000 listings, evaluated offline and online.

Then review your own document as a sceptical senior engineer: list the three riskiest assumptions and how you would test each one in the first two weeks.

#### Solution: what a strong answer covers for option 1

- **Problem and success**: minutes that attendees rate as accurate and complete; action items with owner and date extracted with high precision (wrong owners are worse than missing items); question answering with citations to the meeting and timestamp.
- **Data**: audio recordings; consent and privacy rules (who may be recorded, retention, who may search which meetings); evaluation set of 20 meetings with human-written minutes and action items.
- **Approach**: speech-to-text with speaker diarisation (multimodal lesson); an LLM with structured output for minutes and action items; RAG over transcripts chunked by topic with timestamps for questions.
- **Architecture**: upload, transcription queue, summarisation workflow (chained: segment, summarise per topic, extract actions, consolidate), index with access control per meeting. Long meetings exceed a comfortable context, so summarise per topic and consolidate rather than sending a whole transcript (context engineering lesson).
- **Evaluation**: transcript word error rate on a sample; action-item precision and recall; faithfulness of minutes; retrieval recall on a question set.
- **Risks**: private conversations in recordings; misattributed speakers leading to wrong owners; hallucinated decisions. Mitigate with consent flows, showing speaker labels with confidence, and linking every action item to its transcript timestamp.
- **Operations**: transcription cost per hour of audio, batch overnight for long recordings, monitoring of edit rates on minutes.
- **Riskiest assumptions**: transcription is accurate enough with your accents and jargon (test on five real recordings in week one); attendees accept recording (ask them); action items are said explicitly enough to extract (check against human minutes).

For option 4, a strong answer centres on a popularity baseline and a collaborative-filtering model, a time-based offline split with recall@k, and an A/B test with a primary metric and guardrails (time series, recommenders and graphs lesson). The language model, if any, writes listing descriptions for cold-start items and explains recommendations; it does not do the ranking.

## Reading papers and staying current

### Hands-on: an afternoon's experiment

A new model has been released and your team wants to know whether to switch the documentation assistant to it. The cell below simulates the per-case results of both models on your 80-question eval set (1 is a pass), with latency and cost per request.

1. Compute each model's pass rate and the paired difference.
2. Bootstrap the difference (resample the 80 question indices, 5,000 times) and report a 95% interval, as in the evaluation lesson.
3. List the cases the candidate fixed and the cases it broke.
4. Write the decision in three sentences: what you would do, what trade-off you are accepting, and what would change your mind.

You are done when you can say whether the interval excludes zero and have written a decision that mentions quality, latency and cost.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
difficulty = rng.uniform(0, 1, size=80)          # hidden per-question difficulty
current = (rng.uniform(0, 1, 80) > difficulty * 0.9).astype(int)
candidate = (rng.uniform(0, 1, 80) > difficulty * 0.75).astype(int)
latency = {"current": 1.9, "candidate": 2.6}      # median seconds per request
cost = {"current": 0.011, "candidate": 0.008}     # per request

#### Solution

Resample question indices, not the two models separately, so that each resample compares both models on the same questions.

In [ ]:
boot = np.random.default_rng(0)
idx = boot.integers(0, len(current), size=(5000, len(current)))
diffs = candidate[idx].mean(axis=1) - current[idx].mean(axis=1)
lo, hi = np.percentile(diffs, [2.5, 97.5])
fixed = np.flatnonzero((current == 0) & (candidate == 1))
broke = np.flatnonzero((current == 1) & (candidate == 0))

print(f"current   pass rate {current.mean():.2f}")
print(f"candidate pass rate {candidate.mean():.2f}")
print(f"difference {candidate.mean() - current.mean():+.3f}  "
      f"95% interval [{lo:+.3f}, {hi:+.3f}]")
print(f"fixed {len(fixed)} cases, broke {len(broke)} cases")
print("latency", latency, "cost", cost)

**Decision.** Do not switch the whole assistant yet: the candidate's pass rate is higher, but with 80 questions the interval includes zero, so the quality gain is not established, and the candidate is slower against a latency target we already struggle to meet. We accept paying more per request for now in exchange for predictable latency. Route 20% of traffic to the candidate behind the model interface, add the broken cases to the eval set, and switch if the interval clears zero or if the latency target is relaxed.

## Design document template

Copy this section for each system you design. Fill every heading, even if only with one line; an empty heading is an unanswered question.

### 1. Problem and success criteria

*Who uses it, for what decision, what a mistake costs, and the measurable targets. Is a language model the right tool at all?*

### 2. Data

*Sources, quality, privacy constraints, and where the evaluation set comes from.*

### 3. Approach

*Which rung of the ladder (rules or search, classical ML, prompted LLM, reasoning model, retrieval, workflow, agent, fine-tuning), or which tool from the wider field, and why not the one below it.*

### 4. Architecture

*Components, each model call (tier, prompt, output format, failure behaviour), and which levers you pull: routing, reasoning effort, context engineering and memory, caching, post-training, inference optimisation, hosted versus self-hosted.*

### 5. Evaluation

*Offline set, graders, slices, release thresholds, online metrics.*

### 6. Risks and safeguards

*Injection exposure, data access, consequential actions, privacy, bias. The one failure that would get the system switched off.*

### 7. Operations

*Latency budget, cost per request and per month, monitoring, model upgrades.* Use the estimator from the framework section above.

### 8. Iteration plan

*What ships first, what you will learn from it, what comes next. Then list the three riskiest assumptions and how you will test each in two weeks.*

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/31-system-design.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.